# Phase 1 — Data & Problem Framing

Exploratory Data Analysis, deduplication, and leak-free splits for PlantVillage + PlantDoc

In [ ]:
import sys
sys.path.insert(0, '..')

from src.utils.seed import set_global_seed
from src.utils.config import load_config
from src.data.eda import run_eda, collect_image_paths
from src.data.splits import build_splits, build_plantdoc_split
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

set_global_seed(42)
cfg = load_config('../configs/config.yaml')
print(f"Config loaded. Project: {cfg.project.name}")

## Dataset Overview

This notebook covers two datasets:

| Dataset | Style | Classes | ~Images |
|---|---|---|---|
| **PlantVillage** | Lab-controlled, plain white/grey background | 38 | ~54,000 |
| **PlantDoc** | In-the-wild, realistic farm/field backgrounds | 27 | ~2,600 |

### The Domain Shift Problem

PlantVillage images are photographed against plain white backgrounds under controlled lighting — ideal for annotation but far from real-world conditions. PlantDoc images are taken outdoors on actual farms with complex backgrounds, varying lighting, and occlusion.

> **A model scoring 99% on PlantVillage might score only 60–70% on real farm photos because it learned to key off the plain white background rather than the leaf lesion. We use PlantDoc as an honest real-world benchmark.**

By training on PlantVillage and evaluating on PlantDoc, we can measure how much of the learned signal is genuinely about plant pathology vs. dataset-specific artefacts. This gap is our primary research signal — closing it (via augmentation, domain adaptation, or multi-domain training) is a key Phase 3 objective.

In [ ]:
# Download datasets (run once)
# Requires: pip install kaggle && ~/.kaggle/kaggle.json
# !python ../src/data/download.py --plantvillage --plantdoc
print("Download commands shown above. Skip if data already present.")

PV_ROOT = Path(cfg.paths.data_root)
PD_ROOT = Path(cfg.paths.plantdoc_root)
print(f"PlantVillage expected at: {PV_ROOT} | Exists: {PV_ROOT.exists()}")
print(f"PlantDoc expected at:     {PD_ROOT} | Exists: {PD_ROOT.exists()}")

## PlantVillage EDA

In [ ]:
if PV_ROOT.exists():
    pv_df = run_eda(PV_ROOT, "PlantVillage", check_duplicates=True)
    print(f"\nDataFrame shape: {pv_df.shape}")
    display(pv_df.head())
else:
    print("⚠️  PlantVillage not found. Showing expected structure.")
    print("Expected: data/raw/PlantVillage/<ClassName>/<image>.jpg")

In [ ]:
from IPython.display import Image as IPImage, display as ipy_display
fig_path = Path('../reports/figures/plantvillage_class_distribution.png')
if fig_path.exists():
    ipy_display(IPImage(str(fig_path)))

In [ ]:
fig_path = Path('../reports/figures/plantvillage_sample_grid.png')
if fig_path.exists():
    ipy_display(IPImage(str(fig_path)))

## Duplicate Detection

PlantVillage contains both colour and greyscale versions of many images, as well as near-identical crops. Left unhandled, these create **data leakage**: the same underlying image could appear in both train and test sets, inflating reported accuracy.

**Method:** We use perceptual hashing (`aHash`) with Hamming distance ≤ 8 bits.
- `aHash` (average hash) down-samples each image to 8×8 grayscale pixels and encodes whether each pixel is above/below the mean — producing a 64-bit fingerprint.
- Two images whose fingerprints differ by ≤ 8 bits (out of 64) are considered near-duplicates.
- Within each near-duplicate group we keep exactly **one** representative image.

> **Duplicates are removed BEFORE splitting to prevent data leakage — having the same image in both train and test would inflate test accuracy by an unknown amount.**

The remaining unique images then undergo stratified splitting.

In [ ]:
if 'duplicate_group' in pv_df.columns if 'pv_df' in dir() else False:
    dup_groups = pv_df.groupby('duplicate_group').size()
    n_dup = int((dup_groups[dup_groups > 1] - 1).sum())
    print(f"Near-duplicate groups: {len(dup_groups[dup_groups > 1])}")
    print(f"Redundant images to remove: {n_dup} ({100*n_dup/len(pv_df):.2f}%)")
    
    # Plot group size distribution
    fig, ax = plt.subplots(figsize=(8, 4))
    dup_groups[dup_groups > 1].hist(bins=20, ax=ax, color='coral', edgecolor='white')
    ax.set_xlabel('Images in duplicate group')
    ax.set_ylabel('Number of groups')
    ax.set_title('Near-Duplicate Group Size Distribution')
    plt.tight_layout()
    plt.savefig('../reports/figures/duplicate_groups.png', dpi=150)
    plt.show()
else:
    print("Run EDA with check_duplicates=True to see this section.")

## Building Splits — Methodology

We build **leak-free, stratified** splits using the following four-step pipeline:

| Step | Action | Rationale |
|---|---|---|
| 1 | **Remove corrupted files** | Images that cannot be opened by PIL are excluded from all splits |
| 2 | **Deduplicate** — keep one image per near-duplicate group | Prevents the same visual content appearing in both train and test |
| 3 | **StratifiedShuffleSplit 70 / 15 / 15** | Each class is represented in proportion across train, val, and test |
| 4 | **Validate zero path overlap** | Assert that `set(train) ∩ set(val) = ∅`, etc., across all three splits |

### Expected Split Sizes (approximate, before deduplication)

| Split | Fraction | ~Images |
|---|---|---|
| Train | 70% | ~37,800 |
| Val | 15% | ~8,100 |
| Test | 15% | ~8,100 |

Actual numbers will be slightly lower after removing duplicates and corrupted files.

In [ ]:
manifest = Path('../reports/plantvillage_manifest.csv')
splits_dir = Path('../data/splits')

if manifest.exists():
    splits = build_splits(
        manifest_csv=manifest,
        val_frac=0.15,
        test_frac=0.15,
        seed=42,
        output_dir=splits_dir,
        dataset_prefix='plantvillage',
    )
    
    # Show split sizes
    for name, df in splits.items():
        print(f"{name:5s}: {len(df):6,} images | {df['class_name'].nunique()} classes")
else:
    print("Run EDA first to generate the manifest.")

## PlantDoc — Domain Shift Evaluation

**Domain shift** refers to the degradation in model performance when the deployment distribution differs from the training distribution. Here, that means:

- **Source domain (train):** PlantVillage — controlled lab photography, plain background, uniform lighting
- **Target domain (test):** PlantDoc — outdoor field photos, complex backgrounds, occlusion, varying viewpoints

We do **not** train on PlantDoc. We use it exclusively as a held-out evaluation set to measure how well our model generalises to real-world conditions. A large PlantVillage → PlantDoc accuracy gap signals that the model has overfit to lab artefacts, and that domain adaptation techniques (random background augmentation, style transfer, adversarial training) are warranted in later phases.

Class names between PlantVillage and PlantDoc overlap partially. `build_plantdoc_split` resolves the mapping automatically using the class map exported during the PlantVillage split step.

In [ ]:
if PD_ROOT.exists():
    pd_df = run_eda(PD_ROOT, "PlantDoc", check_duplicates=False)
    
    class_map = splits_dir / 'plantvillage_class_map.json'
    pd_manifest = Path('../reports/plantdoc_manifest.csv')
    
    if class_map.exists() and pd_manifest.exists():
        matched = build_plantdoc_split(pd_manifest, class_map, splits_dir)
        if matched is not None:
            print(f"\nOverlapping classes for domain-shift evaluation:")
            print(matched['class_name'].value_counts().to_string())
else:
    print("PlantDoc not downloaded yet.")

## Class Imbalance Analysis

PlantVillage's 38 classes are **not equally sized**. Some disease categories (e.g. healthy tomato) have many thousands of images while rare conditions have fewer than 400. This matters because:

1. **Accuracy is misleading** — a classifier that always predicts the majority class can achieve high accuracy while failing on rare diseases.
2. **Macro F1 is more honest** — it treats all classes equally regardless of support, exposing failure on minority classes.
3. **Weighted loss** — we pass inverse class-frequency weights to `nn.CrossEntropyLoss` to penalise mistakes on under-represented classes more heavily.

The **imbalance ratio** (max class count / min class count) summarises the severity. Ratios above 10× typically require explicit handling.

In [ ]:
if 'splits' in dir():
    train_df = splits['train']
    class_counts = train_df['class_name'].value_counts()
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Sorted bar chart
    class_counts.sort_values().plot(kind='barh', ax=axes[0], 
                                      color=plt.cm.RdYlGn(
                                          [i/len(class_counts) for i in range(len(class_counts))]))
    axes[0].set_title('Training Set: Images per Class')
    axes[0].set_xlabel('Count')
    
    # Distribution histogram
    axes[1].hist(class_counts.values, bins=20, color='steelblue', edgecolor='white')
    axes[1].set_xlabel('Images per class')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title(f'Class Size Distribution (imbalance ratio: {class_counts.max()/class_counts.min():.1f}x)')
    
    plt.tight_layout()
    plt.savefig('../reports/figures/class_imbalance.png', dpi=150)
    plt.show()
    
    # Inverse class weights (for weighted loss)
    import torch
    n_samples = len(train_df)
    n_classes = train_df['label'].nunique()
    class_counts_arr = train_df['label'].value_counts().sort_index().values
    class_weights = n_samples / (n_classes * class_counts_arr)
    class_weights_tensor = torch.FloatTensor(class_weights)
    print(f"\nClass weight range: {class_weights.min():.3f} — {class_weights.max():.3f}")
    print("These weights will be passed to nn.CrossEntropyLoss(weight=...)")

## Phase 1 Summary

### Key EDA Findings

| Finding | Implication for Modelling |
|---|---|
| PlantVillage has plain-background lab images | Model may overfit to background; use background randomisation augmentation |
| Near-duplicate images present (colour + greyscale pairs) | Must deduplicate **before** splitting to prevent leakage |
| 38 classes, mild-to-moderate imbalance | Use weighted cross-entropy loss + macro F1 as primary metric |
| PlantDoc is in-the-wild with partial class overlap | Use as external domain-shift benchmark, not for training |
| Expected PV→PD accuracy gap: ~20–30 pp | Closing this gap is a key Phase 3 objective |

### Phase 1 Checklist

- [x] Dataset inventory and path validation
- [x] Per-class image count analysis
- [x] Corrupted file removal
- [x] Near-duplicate detection (aHash, Hamming ≤ 8)
- [x] Stratified 70/15/15 splits with zero-overlap validation
- [x] PlantDoc class-map alignment
- [x] Class imbalance quantified; inverse-frequency weights computed
- [x] Split CSVs saved to `data/splits/`
- [x] Figures saved to `reports/figures/`

In [ ]:
print("=" * 60)
print("PHASE 1 SUMMARY")
print("=" * 60)
print()
print("Datasets")
print(f"  PlantVillage: lab-style, clean backgrounds")
print(f"  PlantDoc:     in-the-wild, realistic backgrounds")
print()
print("Split strategy: Stratified 70/15/15 after deduplication")  
print("Data leakage:   Validated — zero path overlap across splits")
print()
print("Next: Type 'continue' to begin Phase 2 (Baselines)")